In [ ]:
import os
os.chdir("C:\\Users\\Sinbad\\Documents\\GitHub\\advanced_security\\report")
print("typst_pyexec working directory: C:\\Users\\Sinbad\\Documents\\GitHub\\advanced_security\\report")


In [ ]:
import os
import timeit
import random
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt


In [ ]:
class RSA:
    """
    RSA implementation for encryption, signature and station-to-station key exchange.
    """

    def __init__(self, private_key: int = None, public_key: int = None, n: int = None, fast_exp: callable = None) -> None:
        self.private_key = private_key
        self.public_key = public_key
        self.n = n
        self.challenge = "Bravo ! Je suis épousplouffé par ta maîtrise du timing attack sur RSA !"
        self.fast_exp = fast_exp if fast_exp is not None else self._default_fast_exp

    def copy(self):
        return RSA(self.private_key, self.public_key, self.n, self.fast_exp)

    def _default_fast_exp(self, y: int, x: int, n: int) -> int:
        """Apply fast exponentiation for y^x modulo n"""
        s = 1
        y %= n

        while x > 0:
            if (x % 2) == 1:
                s = (s * y) % n
            y = (y * y) % n
            x = x >> 1

        return s

    def setKeys(self, private_key: int, public_key: int, n: int) -> None:
        self.private_key = private_key
        self.public_key = public_key
        self.n = n

    def getKeys(self) -> list:
        return [self.public_key, self.n], [self.private_key, self.n]

    def exportPublicKey(self) -> list:
        return [self.public_key, self.n]

    def createKeyPair(self, size: int) -> list:
        p = self.primary_nb_generator(2**(size//2 - 1), 2**(size//2))
        q = self.primary_nb_generator(2**(size//2 - 1), 2**(size//2))
        n, e, d = self.key_generator(p, q, size=size)
        self.setKeys(d, e, n)
        return [n, e, d]

    def encrypt(self, message: int) -> int:
        return self.fast_exp(message, self.public_key, self.n)

    def decrypt(self, cipher: int, private_key: int = None) -> int:
        if private_key is not None:
            return self.fast_exp(cipher, private_key, self.n)
        return self.fast_exp(cipher, self.private_key, self.n)

    def createChallenge(self) -> int:
        challenge_int = int.from_bytes(self.challenge.encode(), 'big')
        self.challenge = self.encrypt(challenge_int)
        return self.challenge

    def decryptChallenge(self, tested_key: int) -> str:
        decrypted_challenge_int = self.decrypt(self.challenge, private_key=tested_key)
        decrypted_challenge_bytes = decrypted_challenge_int.to_bytes((decrypted_challenge_int.bit_length() + 7) // 8, 'big')
        return decrypted_challenge_bytes.decode()

    def fermat_test(self, n: int) -> bool:
        """
        Check if a number is primary by running Fermat test.
        """
        for _ in range(20):
            alpha = random.randint(2, n - 1)
            if self.fast_exp(alpha, n - 1, n) != 1:
                return False
        return True

    def primary_nb_generator(self, a: int, b: int, safe_prime: bool = False) -> int:
        """
        Generate primary number in range a, b

        Parameters
        ----------
        a : int
            lower bound
        b : int
            upper bound
        safe_prime : bool
            Generate primary number p with (p - 1) / 2 also primary

        Returns
        -------
        int
            Primary number generated

        """
        while True:
            p = random.randint(a, b)
            if self.fermat_test(p):
                if safe_prime:
                    if self.fermat_test((p - 1) / 2):
                        return p
                else:
                    return p

    def Euclide(self, a: int, b: int) -> list:
        """
        Euclide's extended algorithm, used to find decryption exponent d for RSA
        Return a list [pgcd(a, b), inverse of a mod b, inverse of b mod a]

        """
        if b > a:
            a, b = b, a  # swap

        r_0, r_1 = a, b
        s_0, s_1 = 1, 0
        t_0, t_1 = 0, 1

        while r_1 != 0:
            q = r_0 // r_1
            r_0, r_1 = r_1, r_0 - q * r_1
            s_0, s_1 = s_1, s_0 - q * s_1
            t_0, t_1 = t_1, t_0 - q * t_1

        return [r_0, s_0 % b, t_0 % a]

    def key_generator(self, p: int = 0, q: int = 0, e: int = 0, size: int = 512) -> list:
        """
        Generate public and private key for RSA
        """
        if p == 0:
            p = self.primary_nb_generator(2**(size//2 - 1), 2**(size//2))
        if q == 0:
            q = self.primary_nb_generator(2**(size//2 - 1), 2**(size//2))

        assert self.fermat_test(p), "p is not primary"
        assert self.fermat_test(q), "q is not primary"
        n = p * q
        phi_n = (p - 1) * (q - 1)

        if e != 0:
            pgcd, _, d = self.Euclide(phi_n, e)
            assert pgcd == 1, "pgcd(phi_n, e) is not equal to 1, so no private key found !"

        else:
            # Find a primary number e with phi_n
            while True:
                e = random.randint(2**(size - 1), 2**(size))
                pgcd, _, d = self.Euclide(phi_n, e)

                # If primary number with phi_n, claim public key
                if pgcd == 1:
                    break
        return n, e, d

def collect_samples(
    RSA_instance: RSA,
    num_samples: int = 1000,
    num_repetitions: int = 10000,
    private_key: int = None,
    progress_bar: bool = True,
    warmup_reps: int = 200,
) -> np.ndarray:
    """Collect timing samples for RSA decryption.

    Each sample is a (ciphertext, min_time_ns) pair. The minimum over
    num_repetitions is used because OS jitter can only ADD latency —
    the minimum is therefore the best estimate of the true computation
    time, as established in the timing attack literature (Kocher 1996,
    Brumley & Boneh 2003).

    Parameters
    ----------
    RSA_instance : RSA
        An instance of the RSA class.
    num_samples : int
        The number of (ciphertext, time) pairs to collect.
    num_repetitions : int
        The number of repetitions per sample. More = better chance of
        one clean, uninterrupted measurement.
    private_key : int, optional
        The private key to use. Defaults to RSA_instance.private_key.
    progress_bar : bool, default=True
        Whether to display a tqdm progress bar.
    warmup_reps : int, default=200
        Warm-up decryptions to stabilize CPU caches and branch predictor.

    Returns
    -------
    np.ndarray
        Array of shape (num_samples, 2): columns are [ciphertext, min_time_ns].
    """
    key = private_key if private_key is not None else RSA_instance.private_key
    n = RSA_instance.n
    decrypt_fn = RSA_instance.decrypt

    # --- Warm-up ---
    # Stabilise CPU caches and branch predictor before any measurement.
    # timeit handles GC disabling automatically.
    warmup_timer = timeit.Timer(
        stmt=lambda: decrypt_fn(random.randint(1, n - 1), private_key=key)
    )
    warmup_timer.timeit(number=warmup_reps)

    # --- Collection ---
    samples = np.empty((num_samples, 2), dtype=np.float64)

    for i in tqdm(range(num_samples), disable=not progress_bar, leave=False):
        cipher = random.randint(1, n - 1)

        timer = timeit.Timer(
            stmt=lambda: decrypt_fn(cipher, private_key=key)
        )

        # repeat=num_repetitions, number=1 → one wall-clock measurement
        # per decryption call, in seconds. timeit disables GC automatically.
        raw_times_ns = np.array(
            timer.repeat(repeat=num_repetitions, number=1)
        ) * 1e9

        # The minimum is the best estimator of true decryption time.
        # OS interrupts can only ADD latency, never subtract it.
        samples[i] = [cipher, raw_times_ns.min()]

    return samples

def timing_attack(
    RSA_instance: RSA,
    num_samples: int = 1000,
    num_repetitions: int = 10000,
    num_iterations: int = 10,
    num_known_bits: int = 5,
    buffer_size: int = 5,
    warmup_reps: int = 200,
    progress_bar: bool = False,
) -> tuple[np.ndarray, list[float]]:
    """
    Perform a timing attack on RSA to recover the private key.

    Uses a beam search over key hypotheses: at each iteration, two
    candidate bits (0 and 1) are tested for each key in the buffer.
    The hypothesis that minimises variance of (server_time - local_time)
    is kept, exploiting the extra multiplication in square-and-multiply
    when a key bit is 1.

    Parameters
    ----------
    RSA_instance : RSA
        An instance of the RSA class.
    num_samples : int
        Number of ciphertext/time pairs to collect per measurement.
    num_repetitions : int
        Repetitions per ciphertext inside collect_samples. More reps
        = better min() estimate. Replaces the old disable_gc flag since
        timeit now handles GC automatically.
    num_iterations : int
        Number of key bits to recover (one per iteration).
    num_known_bits : int
        Number of LSBs of the private key assumed to be known.
    buffer_size : int
        Beam width: how many candidate keys to keep at each iteration.
    warmup_reps : int
        Warm-up decryptions before any measurement (default: 200).
    progress_bar : bool
        Whether to display a progress bar during sample collection.

    Returns
    -------
    tuple[np.ndarray, list[float]]
        candidate_keys : best `buffer_size` recovered key candidates.
        error_rate     : bit error rate (%) for each candidate.
    """
    # --- Collect server reference samples (true private key) ---
    server_samples = collect_samples(
        RSA_instance,
        num_samples=num_samples,
        num_repetitions=num_repetitions,
        warmup_reps=warmup_reps,
        progress_bar=progress_bar,
    )

    # --- Beam search initialisation ---
    initial_key = RSA_instance.private_key & ((1 << num_known_bits) - 1)
    candidate_keys      = np.full(buffer_size, initial_key)
    candidate_variances = np.full(buffer_size, np.inf)

    history_keys      = np.zeros((num_iterations, buffer_size))
    history_variances = np.zeros((num_iterations, buffer_size))

    # --- Bit-by-bit recovery ---
    for i in range(num_iterations):
        tested_keys      = []
        tested_variances = []

        for key in candidate_keys:
            # Hypothesis 0: next bit is 0
            key_h0 = key
            # Hypothesis 1: next bit is 1
            key_h1 = key | (1 << (num_known_bits + i))

            for hyp_key in (key_h0, key_h1):
                hyp_samples = collect_samples(
                    RSA_instance,
                    num_samples=num_samples,
                    num_repetitions=num_repetitions,
                    private_key=hyp_key,
                    warmup_reps=warmup_reps,
                    progress_bar=progress_bar,
                )
                # Variance of time difference: drops when key prefix matches
                # the real key (the extra multiplication aligns in time)
                variance = np.var(server_samples[:, 1] - hyp_samples[:, 1])
                tested_keys.append(hyp_key)
                tested_variances.append(variance)

        tested_keys      = np.array(tested_keys)
        tested_variances = np.array(tested_variances)

        # Keep the buffer_size best hypotheses (lowest variance)
        best_indices        = np.argsort(tested_variances)[:buffer_size]
        candidate_keys      = tested_keys[best_indices]
        candidate_variances = tested_variances[best_indices]

        history_keys[i]      = candidate_keys
        history_variances[i] = candidate_variances

        # --- Per-iteration debug print ---
        best_key       = candidate_keys[0]
        bit_guessed    = (best_key >> (num_known_bits + i)) & 1
        bit_real       = (RSA_instance.private_key >> (num_known_bits + i)) & 1
        h0_var         = tested_variances[tested_keys == key_h0][0]
        h1_var         = tested_variances[tested_keys == key_h1][0]
        correct        = "✓ CORRECT" if bit_guessed == bit_real else "✗ WRONG"
        print(
            f"Iteration {i+1:>{len(str(num_iterations))}}/{num_iterations}: "
            f"bit guessed = {bit_guessed}  "
            f"(var h0 = {h0_var:.4e}, var h1 = {h1_var:.4e})  {correct}"
        )

    # --- Final error rate across all candidates ---
    error_rates = []
    print(f"\n{'─' * 40}")
    for rank, (key, variance) in enumerate(zip(candidate_keys, candidate_variances)):
        guessed_bits = (key >> num_known_bits) % (1 << num_iterations)
        real_bits    = (RSA_instance.private_key >> num_known_bits) % (1 << num_iterations)
        n_errors     = (guessed_bits ^ real_bits).bit_count()
        rate         = n_errors / num_iterations * 100
        error_rates.append(rate)
        print(f"Candidate #{rank+1}")
        print(f"  Key:        {key}")
        print(f"  Variance:   {variance:.4e}")
        print(f"  Error rate: {rate:.2f}%  ({n_errors}/{num_iterations} bits wrong)")
        print(f"{'─' * 40}")

    return candidate_keys, error_rates

def get_samples_stats(
    RSA_instance: RSA,
    d_A: int,
    num_samples: int = 10000,
    warmup_reps: int = 200,
    progress_bar: bool = True,
) -> np.ndarray:
    """
    Collect individual decryption times for a single fixed ciphertext.
    Used to inspect the timing distribution (histogram, min, spread).

    Parameters
    ----------
    RSA_instance : RSA
    d_A : int
        Private key.
    num_samples : int
        Number of individual timing measurements to collect.
    warmup_reps : int
        Warm-up iterations to stabilise caches (default: 200).
    progress_bar : bool
        Show progress bar during measurement.

    Returns
    -------
    np.ndarray shape (num_samples,), times in nanoseconds.
    """
    decrypt_fn = RSA_instance.decrypt
    n = RSA_instance.n

    # Warm-up: timeit handles GC disabling automatically
    warmup_timer = timeit.Timer(
        stmt=lambda: decrypt_fn(random.randint(1, n - 1), private_key=d_A)
    )
    warmup_timer.timeit(number=warmup_reps)

    # One measurement per call (number=1), repeated num_samples times
    # This gives us the full distribution, not a single aggregate
    timer = timeit.Timer(
        stmt=lambda: decrypt_fn(random.randint(1, n - 1), private_key=d_A)
    )

    raw = list(
        tqdm(
            (t * 1e9 for t in timer.repeat(repeat=num_samples, number=1)),
            total=num_samples,
            desc="Sampling",
            disable=not progress_bar,
            leave=False,
        )
    )

    return np.array(raw, dtype=np.float64)

def plot_distributions(samples_stats_h0: np.ndarray, samples_stats_h1: np.ndarray, title: str = "Distribution of decryption times for two hypotheses with branch prediction"):
    plt.figure(figsize=(12, 6))
    plt.suptitle(title)
    plt.subplot(1, 2, 1)
    plt.hist(samples_stats_h0, bins=200)
    plt.axvline(np.min(samples_stats_h0), color='red', linestyle='dashed', linewidth=1, label=f'Min: {np.min(samples_stats_h0):.4f} ms')
    plt.title("Hypothesis h0: the last bit is 0")
    plt.xlabel("Decryption Time")
    plt.ylabel("Frequency")
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.hist(samples_stats_h1, bins=200)
    plt.axvline(np.min(samples_stats_h1), color='red', linestyle='dashed', linewidth=1, label=f'Min: {np.min(samples_stats_h1):.4f} ms')
    plt.title("Hypothesis h1: the last bit is 1")
    plt.xlabel("Decryption Time")
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
#----------------------------------------------------------------------------
# Initialization of RSA parameters for the attack
#----------------------------------------------------------------------------

p_A = 13109499994810966779468866046493465498469807493634236479294124421385342920350717814807375283698575766763256101470694189234369358996750113963585617491399169
q_A = 9497561827984502554523100157901534504433126034087863778629488755692649311435921364240405549590851856701860175924335776598684751639633322074428628372725777
n_A = p_A * q_A
e_A = 4574830074548708213
m_1 = 123456789132456789
d_A = 1685394382767324790326942621450485552187209875614438478305225564629345944620726038114923060947436330701451901921041511234432041036987266468290187679773130363479895993621867708066144608084390089775045890165825736468637468786667820591136139480545376198614216373031208691260339805721685482401743494212035728605

RSA_instance = RSA()
n, e, d = RSA_instance.key_generator(p_A, q_A, e_A)
RSA_instance.setKeys(d, e, n)


In [ ]:
def square_and_multiply(y, x, n):
  s = 1
  y %= n

  while x > 0:
      if (x % 2) == 1:
          s = (s * y) % n # Extra multiplication here when x is odd !
      y = (y * y) % n
      x = x >> 1

  return s


In [ ]:


import gc
import time

def get_samples_stats_manual(
    RSA_instance: RSA,
    d_A: int,
    num_samples: int = 10000,
    disable_gc: bool = True,
    warmup_reps: int = 0,
    progress_bar: bool = True,
) -> np.ndarray:
    """
    Collect decryption times for a fixed ciphertext.

    Parameters
    ----------
    RSA_instance : RSA
    d_A : int
        Private key.
    num_samples : int
    disable_gc : bool
        Disable GC during measurement (default: True).
    warmup_reps : int
        Warm-up iterations before measurement (0 = no warm-up).
    progress_bar : bool
        Show progress bar during measurement (default: True).
    Returns
    -------
    np.ndarray  shape (num_samples,), times in nanoseconds.
    """
    if disable_gc:
        gc.disable()
        gc.collect()

    # Localize hot references
    decrypt_fn = RSA_instance.decrypt
    perf_ns    = time.perf_counter_ns

    # Warm-up
    for _ in tqdm(range(warmup_reps), desc="Warm-up", disable=not progress_bar, leave=False):
        decrypt_fn(d_A)

    # Pre-allocated buffer
    samples = np.empty(num_samples, dtype=np.int64)
    for i in tqdm(range(num_samples), desc="Sampling", disable=not progress_bar, leave=False):
        t0 = perf_ns()
        decrypt_fn(d_A)
        samples[i] = perf_ns() - t0

    if disable_gc:
        gc.enable()

    return samples

samples_stats_0_manual = get_samples_stats_manual(RSA_instance, 0, num_samples=1000, disable_gc=True, warmup_reps=0, progress_bar=False)
samples_stats_1_timeit = get_samples_stats(RSA_instance, 0, num_samples=1000, warmup_reps=0, progress_bar=False)

plt.figure(figsize=(12, 6))
plt.suptitle("Comparison of timing measurements with manual garbage collector (GC) disable versus using timeit (GC handled automatically)")
plt.subplot(1, 2, 1)
plt.hist(samples_stats_0_manual, bins=200)
plt.title("Manual GC disable")
plt.xlabel("Decryption Time (ns)")
plt.ylabel("Frequency")
plt.subplot(1, 2, 2)
plt.hist(samples_stats_1_timeit, bins=200)
plt.title("Using timeit")
plt.xlabel("Decryption Time (ns)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:

big_key = d_A & ((1 << 1000) - 1)
h0_big_key = big_key
h1_big_key = big_key | (1 << 1000)

samples_stats_0_warmup = []
samples_stats_1_warmup = []
samples_stats_0_no_warmup = []
samples_stats_1_no_warmup = []

samples_stats_0_warmup_big_key = []
samples_stats_1_warmup_big_key = []
samples_stats_0_no_warmup_big_key = []
samples_stats_1_no_warmup_big_key = []

for _ in range(20):
    samples_stats_0_warmup.append(np.min(get_samples_stats(RSA_instance, 0, num_samples=1000, warmup_reps=200, progress_bar=False)))
    samples_stats_1_warmup.append(np.min(get_samples_stats(RSA_instance, 1, num_samples=1000, warmup_reps=200, progress_bar=False)))
    samples_stats_0_no_warmup.append(np.min(get_samples_stats(RSA_instance, 0, num_samples=1000, warmup_reps=0, progress_bar=False)))
    samples_stats_1_no_warmup.append(np.min(get_samples_stats(RSA_instance, 1, num_samples=1000, warmup_reps=0, progress_bar=False)))

    samples_stats_0_warmup_big_key.append(np.min(get_samples_stats(RSA_instance, h0_big_key, num_samples=1000, warmup_reps=200, progress_bar=False)))
    samples_stats_1_warmup_big_key.append(np.min(get_samples_stats(RSA_instance, h1_big_key, num_samples=1000, warmup_reps=200, progress_bar=False)))
    samples_stats_0_no_warmup_big_key.append(np.min(get_samples_stats(RSA_instance, h0_big_key, num_samples=1000, warmup_reps=0, progress_bar=False)))
    samples_stats_1_no_warmup_big_key.append(np.min(get_samples_stats(RSA_instance, h1_big_key, num_samples=1000, warmup_reps=0, progress_bar=False)))

plt.figure(figsize=(12, 10))
plt.suptitle("Comparison of differences with (h1) and without (h0) extra multiplications with and without warm-up phase with minimum timing measurements for small and big key sizes")
plt.subplot(2, 1, 1)
plt.title("Small key size")
plt.plot(samples_stats_0_warmup, label="h0 with warm-up")
plt.plot(samples_stats_1_warmup, label="h1 with warm-up")
plt.plot(samples_stats_0_no_warmup, label="h0 without warm-up")
plt.plot(samples_stats_1_no_warmup, label="h1 without warm-up")
plt.xlabel("Number of repetitions")
plt.ylabel("Decryption Time (ns)")
plt.legend()
plt.subplot(2, 1, 2)
plt.title("Big key size")
plt.plot(samples_stats_0_warmup_big_key, label="h0 with warm-up")
plt.plot(samples_stats_1_warmup_big_key, label="h1 with warm-up")
plt.plot(samples_stats_0_no_warmup_big_key, label="h0 without warm-up")
plt.plot(samples_stats_1_no_warmup_big_key, label="h1 without warm-up")
plt.xlabel("Number of repetitions")
plt.ylabel("Decryption Time (ns)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:


def plot_repetition_convergence(
    RSA_instance: RSA,
    d_h0: int,
    d_h1: int,
    sample_counts: list = [1, 10, 100, 500, 1000, 2000, 5000],
    n_runs: int = 10,
):
    h0_means = []
    h1_means = []

    for n in sample_counts:
        tmp_h0 = []
        tmp_h1 = []
        for _ in range(n_runs):
            tmp_h0.append(np.min(get_samples_stats(RSA_instance, d_h0, num_samples=n, warmup_reps=0)))
            tmp_h1.append(np.min(get_samples_stats(RSA_instance, d_h1, num_samples=n, warmup_reps=0)))
        h0_means.append(np.mean(tmp_h0))
        h1_means.append(np.mean(tmp_h1))

    plt.figure(figsize=(6, 6))
    plt.plot(sample_counts, h0_means, label='h0')
    plt.plot(sample_counts, h1_means, label='h1')
    plt.xscale('log')
    plt.xlabel('Number of samples')
    plt.ylabel('Minimum decryption time (ns)')
    plt.title('Convergence of minimum decryption time with number of samples')
    plt.legend()

    plt.tight_layout()
    plt.show()

big_key = d_A & ((1 << 1000) - 1)
h0_big_key = big_key
h1_big_key = big_key | (1 << 1000)

plot_repetition_convergence(RSA_instance, h0_big_key, h1_big_key, sample_counts=[1, 10, 100, 500, 1000, 2000, 3000, 5000, 7000, 10000], n_runs=10)


In [ ]:
def square_and_multiply_with_cost(y, x, n, t_mul=1.1):
    """
    Square-and-multiply algorithm with cost modeling.
     - t_mul: cost factor for multiplication operations
     - t_square: cost factor for squaring operations
    """
    s = 1
    y %= n
    cost = 0

    while x > 0:
        if x & 1:
            s = (s * y) % n
            cost += t_mul * s.bit_count()
        y = (y * y) % n
        x >>= 1
    return s, cost
